## Prepare Workspace

### Import Packages

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

### Set File Paths

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

### Source User Defined Functions/Objects

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Inputs

### Set Estimate Parameters

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
population_weights = df_params[df_params['Type'] == 'population_weights']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]
MOE = 'No'

# view
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Population weights: " + population_weights)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

### Define Import Inputs

In [ ]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = import_tab)
if geography == 'PUMA':
    sample_type = sample_type + '_' + df_inputs['table'].values[0]

if estimate == 'DEC':
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = estimate)
else:
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = sample_type)
# Subset variables
df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set years
years_to_import = list(range(year_start, year_end+1))

def ME_split(text):
    return ",".join(text.split(',')[0:3:2])

if MOE == 'Yes':
    df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)


## For DEC data
if estimate == 'DEC':

    # Reset years to import for DEC
    # Set DEC variables to import
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    years_to_import = [2000, 2010, 2020]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)

## For ACS1 or ACS5 data
if sample_type == 'ACS':
    
    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    # Set tables and variables to import

    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except:
            pass

    if MOE == 'Yes':
        list_vars = ['NAME'] + df_vars['ID'].to_list() + df_vars['Attributes'].to_list()
    else:
        list_vars = ['NAME'] + df_vars['ID'].to_list()
    tables = df_vars['Table'].unique()

    # For tract and county level pull
    if import_tab == 'Counties':
        
        # Import County FIPS mapping
        # Convert to dictionary object for easy state-county combination importing
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        
        df_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values)) 
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = df_fips.copy()
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(tables)
        print(list_vars)

    
    # For MSA level pull
    if import_tab == 'MSA':
    
        # Set MSAs to import
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)


## For PUMS data
if import_tab == 'PUMA':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS on these years)
    # Create dictionary of variable mappings by year (sometimes the variable name changes over time)
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except:
            pass
            
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['PUMA'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS'})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])

    # view
    print(dict_fips)
    print(dict_vars)


# view
df_vars.head(3)

In [ ]:
## For ACS1 or ACS5 data
if sample_type == 'ACS':

    ## Build initial ID fields ##
    print("Building initial ID fields")
    print("")
    
    # this step is done to create the NAME, state, county, tract, and year ID fields for easy left-joining of all variables later
    # (Census Bureau only allows 50 variables to be imported at a time)
          
    # initialize empty list to store data frames
    # pull one table just to get NAME, state, county, tract, and year ID fields
    # combine all years into one data frame

    list_df_acs = []
    
    df_table = df_vars[df_vars['Table'] == tables[0]]
    list_table_vars = ['NAME']
    variables = ",".join(list_table_vars)
    
    # For tract or county level pull
    if import_tab == 'Counties':
        for state in list(dict_fips.keys()):
            print('State: ' + state)
            for year in tqdm(years_to_import):
                try:
                    temp = query_acs(api_Key     = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = variables
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])

                    if geography == 'Tract':
                        temp = temp[['NAME', 'state', 'county', 'tract', 'Year']]
                    if geography == 'County':
                        temp = temp[['NAME', 'state', 'county', 'Year']]
                    list_df_acs.append(temp)
                except:
                    pass
                    
    # For MSA level pull
    if import_tab == 'MSA':
        for year in tqdm(years_to_import):
            try:
                temp =  query_acs(api_Key     = api_key
                                  , estimate  = estimate
                                  , sample    = sample_type
                                  , geography = geography
                                  , variables = variables
                                  , year      = year
                                  , msa       = msa_to_import)
                
                temp = temp[['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']]
                list_df_acs.append(temp)
            except:
                pass
                
    df_acs_raw = pd.concat(list_df_acs)

    # For tracts or counties only
    if import_tab == 'Counties':
        # merge county name onto table
        df_acs_raw = df_acs_raw.merge(df_fips[['County FIPS', 'County Name']], left_on = 'county', right_on = 'County FIPS')
        df_acs_raw.drop(['County FIPS'], axis = 1, inplace = True)
    
        
    print("Finished!")
    print("")
    
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling data from source")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # left join variables onto ID fields for each geography type

    list_df_acs = []
    
    for table in tables:
    
        # keep track of tables being imported
        print("")
        print("Table ID: " + table)
        list_df_tables = []
    
        # import one table at a time
        df_table = df_vars[df_vars['Table'] == table]
        if MOE == 'Yes':
            list_table_vars = [['NAME'] + df_table['ID_Attributes'].to_list()[x:x+5] for x in range(0, len(df_table['ID_Attributes'].to_list()), 5)]
        else:
            list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+25] for x in range(0, len(df_table['ID'].to_list()), 25)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        # pull all years and counties for each table
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_tables = []

            # Tract/County level inputs
            if import_tab == 'Counties':
                for state in list(dict_fips.keys()):
                    print('State: ' + state)
                    for year in tqdm(years_to_import):
                        try:
                            list_df_tables.append(
                                query_acs(api_Key     = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , state     = state
                                          , county    = dict_fips[state])
                            )
                        except:
                            pass
                            
            # MSA level inputs
            if import_tab == 'MSA':
                for year in tqdm(years_to_import):
                    try:
                        list_df_tables.append(
                            query_acs(api_Key     = api_key
                                      , estimate  = estimate
                                      , sample    = sample_type
                                      , geography = geography
                                      , variables = variables
                                      , year      = year
                                      , msa       = msa_to_import)
                        )
                    except:
                        pass
        
        df_temp = pd.concat(list_df_tables)

        if geography == 'Tract':
            df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'left')
        if geography == 'County':
            df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'Year'], how = 'left')
        if geography == 'MSA':
            df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'left')

    
    print("Finished!")


## For DEC data
if estimate == 'DEC':

    ## Build initial ID fields ##
    print("Building initial ID fields")
    print("")
          
    # this step is done to create the NAME, state, county, tract, and year ID fields for easy left-joining of all variables later
    # (Census Bureau only allows 50 variables to be imported at a time)
          
    # initialize empty list to store data frames
    # pull one table just to get NAME, state, county, tract, and year ID fields
    # combine all years into one data frame    
    
    list_df_acs = []
    
    # For tract level pull
    if import_tab == 'Counties':
        for state in list(dict_fips.keys()):
            print('State: ' + state)
            for year in tqdm(years_to_import):
                try:
                    temp = query_acs(api_Key     = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = ','.join(dict_vars[str(year)])
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
                    
                    if geography == 'Tract':
                        temp = temp[['NAME', 'state', 'county', 'tract', 'Year']]
                    if geography == 'County':
                        temp = temp[['NAME', 'state', 'county', 'Year']]
                    list_df_acs.append(temp)   
                except:
                    pass

    # For MSA level pull
    if import_tab == 'MSA':
        for year in tqdm(years_to_import):
            try:
                temp = query_acs(api_Key     = api_key
                                 , estimate  = estimate
                                 , sample    = sample_type
                                 , geography = geography
                                 , variables = ','.join(dict_vars[str(year)])
                                 , year      = year
                                 , state     = state
                                 , msa       = msa_to_import)
                 
                temp = temp[['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']]
                list_df_acs.append(temp)  
            except:
                pass

    df_acs_raw = pd.concat(list_df_acs)

    # For tracts or counties only
    if import_tab == 'Counties':
        # merge county name onto table
        df_acs_raw = df_acs_raw.merge(df_fips[['County FIPS', 'County Name']], left_on = 'county', right_on = 'County FIPS')
        df_acs_raw.drop(['County FIPS'], axis = 1, inplace = True)
    
        
    print("Finished!")
    print("")


    print("Importing and compiling data from source")
    print("")

    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties
    # left join variables onto ID fields for each geography type
    
    list_df_acs = []
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                list_df_acs.append(
                    query_acs(api_Key      = api_key
                               , estimate  = estimate
                               , sample    = sample_type
                               , geography = geography
                               , variables = ','.join(dict_vars[str(year)])
                               , year      = year
                               , state     = state
                               , county    = dict_fips[state])
                )
                
            except:
                pass
                
    df_temp = pd.concat(list_df_acs)
    
    if geography == 'Tract':
        df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'left')
    if geography == 'County':
        df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'Year'], how = 'left')
    if geography == 'MSA':
        df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'left')
    
    print("Finished!")


## For PUMS tables
if geography == 'PUMA':
    
    print("Importing and compiling data from source")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and PUMAs
    # import all variables
    # combine all years and counties
    # left join variables onto ID fields for each geography type
    
    list_df_acs = []
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                df_pums = query_acs(api_Key      = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = ','.join(dict_vars[str(year)])+',SERIALNO'
                                     , year      = year
                                     , state     = state
                                     , puma = dict_fips[state])
                df_pums = df_pums.drop(['public use microdata area'], axis = 1)
                # df_pums.columns = dict_vars[str(np.max(years_to_import))] + ['SERIALNO', 'RT', 'state', 'Year']
                df_pums.columns = dict_vars[str(np.max(years_to_import))] + ['SERIALNO', 'state', 'Year']
                df_pums['state'] = state
                
                list_df_acs.append(df_pums)
                
            except:
                pass
                
    df_acs_raw = pd.concat(list_df_acs)

    print("Finished!")

In [ ]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_acs_raw.shape)
print(df_acs_raw.Year.unique())
df_acs_raw.head(3)

In [ ]:
## Make copy of data frame
df_acs = df_acs_raw.copy()

## For tract level data
if geography == 'Tract':

    # Melt data from wide to long
    # Convert imported values to numeric
    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    # Manually check column names and clean as needed+

    df_acs = df_acs.replace('-666666666', np.nan)
    df_acs = df_acs.replace('null', np.nan)
    df_acs = df_acs.dropna(axis = 1, how = 'all')
    
    df_acs = pd.melt(df_acs
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'tract', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    df_acs = df_acs.dropna()
    
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    df_acs = df_acs.merge(df_vars[['ID', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    df_acs = df_acs[['ID', 'County Name', 'NAME', 'state', 
                     'county', 'tract', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    df_acs = df_acs.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
        , 'tract':'Tract ID'
    })

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    if population_weights == 'Yes':
        df_pop = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3', 'Pop_3 Tract ACS5 Long.xlsx'), sheet_name = 'Tracts')
        df_pop = df_pop[['NAME', 'Year','Race_Ethnicity', 'Population']]
        conditions = [
                        (df_pop["Race_Ethnicity"] == 'All'                                            ),
                        (df_pop["Race_Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                        (df_pop["Race_Ethnicity"] == 'Asian (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Black or African American (NH)'                 ),
                        (df_pop["Race_Ethnicity"] == 'Hispanic or Latino'                             ),
                        (df_pop["Race_Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                        (df_pop["Race_Ethnicity"] == 'White (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Some other race (NH)'                           ),
                        (df_pop["Race_Ethnicity"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        df_acs = df_acs.merge(df_pop, on = ['NAME', 'Year', 'Race_Ethnicity'], how = 'left')
        # df_acs = df_acs.fillna(0)
        df_acs = df_acs.dropna()
        

## For County level data
if geography == 'County':

    # Clean missings
    # Melt data from wide to long
    # Convert imported values to numeric
    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    # Manually check column names and clean as needed
    
    df_acs = df_acs.replace('null', np.nan)
    # df_acs = df_acs.dropna(axis = 0, how = "any") # this seems wrong to me.  not sure why i'd do this
    df_acs = df_acs.replace('-666666666', np.nan)
    df_acs = df_acs.dropna(axis = 1, how = 'all')
    
    df_acs = pd.melt(df_acs
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    df_acs = df_acs.merge(df_vars[['ID', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    df_acs = df_acs[['ID', 'County Name', 'NAME', 'state', 
                       'county', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    df_acs = df_acs.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
    })

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    if population_weights == 'Yes':
        df_pop = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3', 'Pop_3 Tract ACS5 Long.xlsx'), sheet_name = 'Counties')
        df_pop = df_pop[['County Name', 'Year','Race_Ethnicity', 'Population']]
        conditions = [
                        (df_pop["Race_Ethnicity"] == 'All'                                            ),
                        (df_pop["Race_Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                        (df_pop["Race_Ethnicity"] == 'Asian (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Black or African American (NH)'                 ),
                        (df_pop["Race_Ethnicity"] == 'Hispanic or Latino'                             ),
                        (df_pop["Race_Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                        (df_pop["Race_Ethnicity"] == 'White (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Some other race (NH)'                           ),
                        (df_pop["Race_Ethnicity"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        df_acs = df_acs.merge(df_pop, on = ['County Name', 'Year', 'Race_Ethnicity'], how = 'left')
        df_acs = df_acs.dropna()


## For MSA level data
if geography == 'MSA':

    # Rename columns, make sure MSA titles are consitent with current year (sometimes the MSA name changes over time)
    # Melt data from wide to long
    # Convert imported values to numeric
    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    # Manually check column names and clean as needed
    
    df_acs = df_acs.replace('-666666666', np.nan)
    df_acs = df_acs.replace('null', np.nan)
    df_acs = df_acs.dropna(axis = 1, how = 'all')

    
    df_acs = df_acs.rename(columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})
    df_msa_map = df_acs[df_acs['Year'] == 2022][['NAME', 'MSA_ID']].drop_duplicates().rename(columns = {'NAME':'MSA'})
    df_acs = df_acs.merge(df_msa_map, on = 'MSA_ID', how = 'left')
    df_acs = pd.melt(df_acs
                      , id_vars = ['NAME', 'MSA', 'MSA_ID', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )

    df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)
    df_acs = df_acs.merge(df_vars[['ID', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    df_acs = df_acs[['ID', 'MSA_ID', 'MSA', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    if population_weights == 'Yes':
        df_pop = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3', 'Pop_3 MSA ACS5 Long.xlsx'), sheet_name = 'MSA')
        df_pop = df_pop[['MSA', 'Year','Race_Ethnicity', 'Population']]
        conditions = [
                        (df_pop["Race_Ethnicity"] == 'All'                                            ),
                        (df_pop["Race_Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                        (df_pop["Race_Ethnicity"] == 'Asian (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Black or African American (NH)'                 ),
                        (df_pop["Race_Ethnicity"] == 'Hispanic or Latino'                             ),
                        (df_pop["Race_Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                        (df_pop["Race_Ethnicity"] == 'White (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Some other race (NH)'                           ),
                        (df_pop["Race_Ethnicity"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        df_acs = df_acs.merge(df_pop, on = ['MSA', 'Year', 'Race_Ethnicity'], how = 'left')
        df_acs = df_acs.dropna()



if geography == 'PUMA':

    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data
    df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

    # ## For non-integers only
    # for group in df_inputs['groups'].dropna().values:
    #     df_acs[group] = df_acs[group].astype(str).apply('{:0>2}'.format)
        
    # df_vars['Value1'] = df_vars['Value1'].astype(str).apply('{:0>2}'.format)
    # df_acs ['state' ] = df_acs ['state' ].astype(str).apply('{:0>2}'.format)
    
    # df_acs[df_vars['Suggested Weight'].values[0]] = df_acs[df_vars['Suggested Weight'].values[0]].astype(int)
    # df_acs[list(df_inputs['groups'].dropna().values)] = df_acs[list(df_inputs['groups'].dropna().values)].astype("string")

    # df_puma_vars = df_vars.pivot_table(index = ['Year', 'Value1']
    #                                        , columns = 'ID2'
    #                                        , values = 'Description2'
    #                                        , aggfunc = lambda x: x).reset_index()

    # cols = ['Year', 'Value1'] + list(df_inputs['groups'].dropna().values)
    # df_puma_vars = df_puma_vars[cols]

    # list_values = []
    # for group in df_inputs['groups'].dropna().values:
    #     list_values = list_values + list(df_acs[group].values)
    # set_values = set(list_values)
    
    # df_puma_vars = df_puma_vars[df_puma_vars['Value1'].isin(set_values)]
    # df_puma_vars = df_puma_vars.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1', 'Year_desc':'Year'})

    # for col in cols[2:]:
    #     df_acs = df_acs.merge(df_puma_vars[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'inner')
    #     df_acs = df_acs.drop(['Value1'], axis = 1)
    # ## For non-integers only

## TODO: 
# Need to separate out integer fields (do not need to merge description) vs non-integer fields (do need to merge description) using "Data Type" field
# Need to link persons weighted file to households weighted file using SERIALNO
# Need to test if persons file can have more than one HISP/RAC1P per SERIALNO

# view
df_acs.head(3)

In [ ]:
MOE = 'Yes'
if MOE == 'Yes':
    path_out_me = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
    df_acs_me = df_acs[df_acs['Year'].isin([2009, 2013, 2018, 2022])]
    if geography == 'MSA':
        df_me = df_acs_me[df_acs_me['Variable'].isna()]
        df_acs_me = df_acs_me.dropna()
        df_me = df_me[['ID', 'MSA_ID', 'Year', 'Total']].rename(columns = {'Total':'ME (+/-)'})
        df_me['ID'] = df_me['ID'].apply(lambda s : re.sub("M", "E", s))
        df_acs_me = df_acs_me.merge(df_me, on = ['ID', 'MSA_ID', 'Year'], how = 'left')
        df_acs_me.to_excel(os.path.join(path_out_me, "".join([indicator_name, '_', geography, '_ME_', estimate, '.xlsx'])), index = False)

    if geography == 'County':
        df_me = df_acs_me[df_acs_me['Variable'].isna()]
        df_acs_me = df_acs_me.dropna()
        df_me = df_me[['Table ID', 'County Name', 'Year', 'Total']].rename(columns = {'Total':'ME (+/-)'})
        df_me['Table ID'] = df_me['Table ID'].apply(lambda s : re.sub("M", "E", s))
        df_acs_me = df_acs_me.merge(df_me, on = ['Table ID', 'County Name', 'Year'], how = 'left')
        df_acs_me.to_excel(os.path.join(path_out_me, "".join([indicator_name, '_', geography, '_ME_', estimate, '.xlsx'])), index = False)
        

In [ ]:
## For ACS tables
if sample_type == 'ACS':
    
    # Create "Categorical" race/ethnicity field for sorting
    # Sort by geography, variable mapping, and race/ethnicity
    # sort and then remove categorical field

    df_acs['Race_Ethnicity_sort'] = pd.Categorical(df_acs['Race_Ethnicity'], ['All'
                                                                     , 'American Indian or Alaska Native'
                                                                     , 'American Indian or Alaska Native (NH)'
                                                                     , 'Asian'
                                                                     , 'Asian (NH)'
                                                                     , 'Black or African American'
                                                                     , 'Black or African American (NH)'
                                                                     , 'Hispanic or Latino'
                                                                     , 'Native Hawaiian or other Pacific Islander'
                                                                     , 'Native Hawaiian or other Pacific Islander (NH)'
                                                                     , 'White'
                                                                     , 'White (NH)'
                                                                     , 'Some other race'
                                                                     , 'Some other race (NH)'
                                                                     , 'Two or more races'
                                                                     , 'Two or more races (NH)'
                                                                             ])
    if geography == 'Tract':
        df_acs = df_acs.sort_values(by = ['NAME', 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = [True, False, True, True])
    if geography == 'MSA':
        df_acs = df_acs.sort_values(by = ['MSA', 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = [True, False, True, True])
    df_acs = df_acs.drop(['Race_Ethnicity_sort', 'Sort'], axis = 1)


## For PUMS tables
if geography == 'PUMA':

    # Remove rows with missing values
    # Sort by PUMA, Year, then by each group
    # Only keep description mappings, remove the original PUMS values
    # Rollup using suggested weight field
    # TODO: merge on PUMA name field

    df_acs = df_acs.dropna()
    groups = list(df_inputs['groups'].dropna().values)
    
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums[df_fips_pums['STATEFP'].isin(list(dict_fips.keys()))]
    df_fips_pums = df_fips_pums[['PUMA5CE', 'PUMA NAME', 'Years']].rename(columns = {'PUMA5CE':'PUMA'}).drop_duplicates()

    df_acs1 = df_acs[df_acs['Year'].isin(sequence(2012, 2021, 1))]
    df_acs2 = df_acs[df_acs['Year'].isin(sequence(2022, 2031, 1))]
    
    df_acs1 = df_acs1.merge(df_fips_pums[df_fips_pums['Years'] == '2012-2021'], on = 'PUMA', how = 'left')
    df_acs2 = df_acs2.merge(df_fips_pums[df_fips_pums['Years'] == '2022-2031'], on = 'PUMA', how = 'left')
    df_acs = pd.concat([df_acs1, df_acs2])
    df_acs = df_acs.drop('Years', axis = 1)
    
    df_acs = df_acs.sort_values(
        ['PUMA', 'Year'] + groups
        , ascending = [True, False] + [item in groups for item in groups]
    )

    # ## For non-integers only
    # df_acs = df_acs.drop(groups, axis = 1)

    # if 'HISP' in groups:
    #     df_acs.loc[df_acs['HISP_desc'     ] == 'Hispanic or Latino', 'RAC1P_desc'     ] = 'Hispanic or Latino'
    #     df_acs = df_acs.drop('HISP_desc', axis = 1)

    # if 'HHLDRHISP' in groups:
    #     df_acs.loc[df_acs['HHLDRHISP_desc'] == 'Hispanic or Latino', 'HHLDRRAC1P_desc'] = 'Hispanic or Latino'
    # ## For non-integers only
    
    df_acs = df_acs.groupby(
        list(df_acs.drop([df_vars['Suggested Weight'].values[0]], axis = 1).columns), as_index = False, sort = False
    )[df_vars['Suggested Weight'].values[0]].agg(sum)

    col = df_acs.pop('PUMA NAME')
    df_acs.insert(1, 'PUMA NAME', col)

    col = df_acs.pop('state')
    df_acs.insert(0, 'state', col)

# View    
df_acs.head(3)

In [ ]:
if geography == 'Tract':
    
    # Create MPO and MSA groupings
    # Merge groupings
    # reorder columns
    # fill missing values (represent a population of 0)
    df_mpo = df_fips[['County Name', 'MPO']]
    df_acs = df_acs.merge(df_mpo, on = ['County Name'], how = 'left')
    if population_weights == 'Yes':
        cols = ['Table ID', 'State FIPS', 'MPO', 'County Name',
                'County FIPS', 'Tract ID', 'NAME', 'Year', 'Variable', 'Race_Ethnicity', 'Total', 'Population']
    else:
        cols = ['Table ID', 'State FIPS', 'MPO', 'County Name', 
            'County FIPS', 'Tract ID', 'NAME', 'Year', 'Variable', 'Race_Ethnicity', 'Total']
    df_acs = df_acs[cols]
    df_acs['Total'] = df_acs['Total'].fillna(0)
    

    # If we need metrics weighted by population
    if population_weights == 'Yes':

        # Fill missings with 0, then 1 to make sure nothing gets removed if population is 0
        # create weighted average lambda function
        # roll up to different geographies using population weighted average
        
        df_acs['Population'] = df_acs['Population'].fillna(0)
        df_acs['Population'] = df_acs['Population'].replace(0, 1)
        wm = lambda x: np.average(x, weights = df_acs.loc[x.index, "Population"])
        df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 
                                  'Variable', 'Year', 'Race_Ethnicity'
                                 ], as_index = False, sort = True).agg(Population = ('Population', 'sum'), Total = ('Total', wm))  
        df_acs1 = df_acs1.sort_values(['NAME', 'Year'], ascending = [True, False])
        df_counties1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 
                                       'Variable', 'Year', 'Race_Ethnicity'
                                      ], as_index = False, sort = True).agg(Population = ('Population', 'sum'), Total = ('Total', wm))  
        df_counties1 = df_counties1.sort_values(['County FIPS', 'Year'], ascending = [True, False])
        df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                                  'Variable', 'Year', 'Race_Ethnicity'
                                 ], as_index = False, sort = True).agg(Population = ('Population', 'sum'), Total = ('Total', wm))  
        df_mpo1 = df_mpo1.sort_values(['MPO', 'Year'], ascending = [True, False])


        
    else:
        
        # If no weighting is needed
        
        df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 
                                  'Variable', 'Year', 'Race_Ethnicity'
                                 ], as_index = False, sort = False)['Total'].sum()
        df_acs1 = df_acs1.sort_values(['NAME', 'Year'], ascending = [True, False])
        df_counties1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 
                                       'Variable', 'Year', 'Race_Ethnicity'
                                      ], as_index = False, sort = False)['Total'].sum()
        df_counties1 = df_counties1.sort_values(['County FIPS', 'Year'], ascending = [True, False])
        df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                                  'Variable', 'Year', 'Race_Ethnicity'
                                 ], as_index = False, sort = False)['Total'].sum()
        df_mpo1 = df_mpo1.sort_values(['MPO', 'Year'], ascending = [True, False])

    # Reshape data to wide format
    df_acs2 = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID'
                                           , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_acs2 = df_acs2.sort_values(['NAME', 'Year'], ascending = [True, False])
    df_counties2 = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS',
                                            'County Name', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_counties2 = df_counties2.sort_values(['County FIPS', 'Year'], ascending = [True, False])
    df_mpo2 = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_mpo2 = df_mpo2.sort_values(['MPO', 'Year'], ascending = [True, False])

    # missing values represent a population of 0
    df_acs2 = df_acs2.fillna(0)
    df_acs2 = df_acs2.fillna(0)
    df_mpo2 = df_mpo2.fillna(0)


    
    ## Check if we want to calculate proportions
    if percentages == 'Yes':

        if num_vars == 1:
            df_acs1     ['Percentage'] = 100*df_acs1     ['Total'] / df_acs1     [df_acs1     ['Race_Ethnicity'] != 'All'].groupby(['NAME'      , 'Year'               ])['Total'].transform('sum')
            df_counties1['Percentage'] = 100*df_counties1['Total'] / df_counties1[df_counties1['Race_Ethnicity'] != 'All'].groupby(['State FIPS', 'County FIPS', 'Year'])['Total'].transform('sum')
            df_mpo1     ['Percentage'] = 100*df_mpo1     ['Total'] / df_mpo1     [df_mpo1     ['Race_Ethnicity'] != 'All'].groupby(['MPO'       , 'Year'               ])['Total'].transform('sum')
        if num_vars > 1:
            df_acs1     ['Percentage'] = 100*df_acs1     ['Total'] / df_acs1     .groupby(['NAME'      , 'Year'               , 'Race_Ethnicity'])['Total'].transform('sum')
            df_counties1['Percentage'] = 100*df_counties1['Total'] / df_counties1.groupby(['State FIPS', 'County FIPS', 'Year', 'Race_Ethnicity'])['Total'].transform('sum')
            df_mpo1     ['Percentage'] = 100*df_mpo1     ['Total'] / df_mpo1     .groupby(['MPO'       , 'Year'               , 'Race_Ethnicity'])['Total'].transform('sum')

        # Reshape data to wide format
        df_acs2_perc = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID', 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_acs2_perc = df_acs2_perc.sort_values(['NAME', 'Year'], ascending = [True, False])
        df_counties2_perc = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS', 'County Name', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_counties2_perc = df_counties2_perc.sort_values(['County FIPS', 'Year'], ascending = [True, False])
        df_mpo2_perc = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_mpo2_perc = df_mpo2_perc.sort_values(['MPO', 'Year'], ascending = [True, False])
        
        # missing values represent a population of 0
        df_acs2_perc = df_acs2_perc.fillna(0)
        df_acs2_perc = df_acs2_perc.fillna(0)
        df_mpo2_perc = df_mpo2_perc.fillna(0)


if geography == 'County':
    # Create MPO and MSA groupings
    # Merge groupings
    # reorder columns
    # fill missing values (represent a population of 0)
    
    df_mpo = df_fips[['County Name', 'MPO']]    
    df_acs = df_acs.merge(df_mpo, on = ['County Name'], how = 'left')
    cols = ['Table ID', 'Table Name', 'Label', 'State FIPS', 'MPO', 'County Name', 
            'County FIPS', 'NAME', 'Year', 'Variable', 'Race_Ethnicity', 'Total']
    df_acs = df_acs[cols]
    df_acs['Total'] = df_acs['Total'].fillna(0)

    
    # Groupings roll up
    df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'NAME', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()    
    df_acs1 = df_acs1.sort_values(['County FIPS', 'Year'], ascending = [True, False])
    df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()
    df_mpo1 = df_mpo1.sort_values(['MPO', 'Year'], ascending = [True, False])

    # Reshape data to wide format
    df_acs2 = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS'
                                           , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_acs2 = df_acs2.sort_values(['County FIPS', 'Year'], ascending = [True, False])
    df_mpo2 = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_mpo2 = df_mpo2.sort_values(['MPO', 'Year'], ascending = [True, False])
    
    # missing values represent a population of 0
    df_acs2 = df_acs2.fillna(0)
    df_mpo2 = df_mpo2.fillna(0)


    
    ## Check if we want to calculate proportions
    if percentages == 'Yes':

        if num_vars == 1:
            df_acs1['Percentage'] = 100*df_acs1['Total'] / df_acs1[df_acs1['Race_Ethnicity'] != 'All'].groupby(['NAME', 'Year'])['Total'].transform('sum')
            df_mpo1['Percentage'] = 100*df_mpo1['Total'] / df_mpo1[df_mpo1['Race_Ethnicity'] != 'All'].groupby(['MPO' , 'Year'])['Total'].transform('sum')
        if num_vars > 1:
            df_acs1['Percentage'] = 100*df_acs1['Total'] / df_acs1.groupby(['NAME', 'Year', 'Race_Ethnicity'])['Total'].transform('sum')
            df_mpo1['Percentage'] = 100*df_mpo1['Total'] / df_mpo1.groupby(['MPO' , 'Year', 'Race_Ethnicity'])['Total'].transform('sum')

        # Reshape data to wide format
        df_acs2_perc = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS',  'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_acs2_perc = df_acs2_perc.sort_values(['County FIPS', 'Year'], ascending = [True, False])
        df_mpo2_perc = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_mpo2_perc = df_mpo2_perc.sort_values(['MPO', 'Year'], ascending = [True, False])
        
        # missing values represent a population of 0
        df_acs2_perc = df_acs2_perc.fillna(0)
        df_mpo2_perc = df_mpo2_perc.fillna(0)


if geography == 'MSA':

    # Groupings roll up
    df_msa1 = df_acs.groupby(['MSA', 
                              'Variable', 'Year', 'Race_Ethnicity'
                             ], as_index = False, sort = False)['Total'].sum()
    df_msa1 = df_msa1.sort_values(['MSA', 'Year'], ascending = [True, False])

    # Reshape data to wide format
    df_msa2 = df_msa1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_msa2 = df_msa2.sort_values(['MSA', 'Year'], ascending = [True, False])

    # missing values represent a population of 0
    df_msa2 = df_msa2.fillna(0)

    ## Check if we want to calculate proportions
    if percentages == 'Yes':

        # Estimate proportions by groupings
        if num_vars == 1:
            df_msa1['Percentage'] = 100*df_msa1['Total'] / df_msa1[df_msa1['Race_Ethnicity'] != 'All'].groupby(['MSA', 'Year'])['Total'].transform('sum')
        if num_vars > 1:
            df_msa1['Percentage'] = 100*df_msa1['Total'] / df_msa1.groupby(['MSA', 'Year', 'Race_Ethnicity'])['Total'].transform('sum')
            
        # Reshape data to wide format
        df_msa2_perc = df_msa1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Percentage').reset_index()
        
print("Done!  Probably good")

In [ ]:
if indicator_name == 'Income_3':
    df_mpo_wm = df_mpo1[df_mpo1['Race_Ethnicity'] == 'All'].reset_index(drop = True)

    df_acs1      = df_acs1     .merge(df_mpo_wm[['Year', 'Total']].rename(columns = {'Total':'Regional Median Household Income'}), on = ['Year'], how = 'left')
    df_counties1 = df_counties1.merge(df_mpo_wm[['Year', 'Total']].rename(columns = {'Total':'Regional Median Household Income'}), on = ['Year'], how = 'left')
    df_mpo1      = df_mpo1     .merge(df_mpo_wm[['Year', 'Total']].rename(columns = {'Total':'Regional Median Household Income'}), on = ['Year'], how = 'left')

    df_acs1     ['Percent of Regional Median Household Income'] = df_acs1     ['Total']/df_acs1     ['Regional Median Household Income']
    df_counties1['Percent of Regional Median Household Income'] = df_counties1['Total']/df_counties1['Regional Median Household Income']
    df_mpo1     ['Percent of Regional Median Household Income'] = df_mpo1     ['Total']/df_mpo1     ['Regional Median Household Income']

    df_acs1      = df_acs1     .rename(columns = {'Total':'Median Household Income'})
    df_counties1 = df_counties1.rename(columns = {'Total':'Median Household Income'})
    df_mpo1      = df_mpo1     .rename(columns = {'Total':'Median Household Income'})


if indicator_name in ['Cost_5', 'Income_2', 'Broadband_2']:
    if geography == 'Tract':
        df_acs1      = df_acs1     .rename(columns = {'Total':'Households'})
        df_counties1 = df_counties1.rename(columns = {'Total':'Households'})
        df_mpo1      = df_mpo1     .rename(columns = {'Total':'Households'})
    if geography == 'MSA':
        df_msa1 = df_msa1.rename(columns = {'Total':'Households'})


if indicator_name in ['Pop_3', 'Pop_4', 'Edu_1', 'Labor_1', 'Health_2', 'Income_4', 'Commute_1']:
    if geography == 'Tract':
        df_acs1      = df_acs1     .rename(columns = {'Total':'Population'})
        df_counties1 = df_counties1.rename(columns = {'Total':'Population'})
        df_mpo1      = df_mpo1     .rename(columns = {'Total':'Population'})
    if geography == 'MSA':
        df_msa1 = df_msa1.rename(columns = {'Total':'Population'})


if geography == 'PUMA':
    if 'HISP_desc' in df_acs.columns:
        df_acs = df_acs.drop(['HISP_desc'], axis = 1)



In [ ]:
## For csv files:
# remove state FIPS field
# remove "All" category for race/ethnicity
# make sure index is removed
# make sure proportions are now percentages

if geography == 'Tract':
    df_acs1_csv      = df_acs1     .drop(['State FIPS', 'County FIPS', 'Tract ID'], axis = 1)
    df_counties1_csv = df_counties1.drop(['State FIPS', 'County FIPS'], axis = 1)
    df_mpo1_csv      = df_mpo1     .drop(['State FIPS'               ], axis = 1)
    
    if len(unique(df_acs1.Race_Ethnicity.values)) > 1:
        df_acs1_csv      = df_acs1_csv     [df_acs1_csv     ['Race_Ethnicity'] != 'All']
        df_counties1_csv = df_counties1_csv[df_counties1_csv['Race_Ethnicity'] != 'All']
        df_mpo1_csv      = df_mpo1_csv     [df_mpo1_csv     ['Race_Ethnicity'] != 'All']
    
    df_acs1_csv      = df_acs1_csv     .reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    df_counties1_csv = df_counties1_csv.reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    df_mpo1_csv      = df_mpo1_csv     .reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    
    df_acs1_csv     .columns = [col.lower() for col in df_acs1_csv     .columns]
    df_counties1_csv.columns = [col.lower() for col in df_counties1_csv.columns]
    df_mpo1_csv     .columns = [col.lower() for col in df_mpo1_csv     .columns]
    
    
    df_acs1_csv     .columns = [re.sub('[\s+]', '_', col.strip()) for col in df_acs1_csv     .columns]
    df_counties1_csv.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_counties1_csv.columns]
    df_mpo1_csv     .columns = [re.sub('[\s+]', '_', col.strip()) for col in df_mpo1_csv     .columns]

if geography == 'MSA':
    
    if len(unique(df_msa1.Race_Ethnicity.values)) > 1:
        df_msa1_csv = df_msa1[df_msa1['Race_Ethnicity'] != 'All']
    else:
        df_msa1_csv = df_msa1.copy()
        
    df_msa1_csv  = df_msa1_csv.reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    df_msa1_csv.columns = [x.lower() for x in df_msa1_csv.columns]
    df_msa1_csv.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa1_csv.columns]


if geography == 'PUMA':
    df_acs.columns = [re.sub('_desc', '', col) for col in df_acs.columns]
    df_acs_csv = df_acs.rename(columns = {'Year':'year', 'state': 'State FIPS'})



In [ ]:
# Set output name for .xlsx files
name_output_long_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' Long.xlsx']
name_output_wide_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' Wide.xlsx']

name_output_long_xlsx = "".join(name_output_long_xlsx)
name_output_wide_xlsx = "".join(name_output_wide_xlsx)

# Set output name for .csv files

if geography == 'Tract':
    name_output_tract_csv  = [indicator_name, '_Tract_' , estimate, '.csv']
    name_output_county_csv = [indicator_name, '_County_', estimate, '.csv']
    name_output_MPO_csv    = [indicator_name, '_MPO_'   , estimate, '.csv']
    name_output_tract_csv  = "".join(name_output_tract_csv )
    name_output_county_csv = "".join(name_output_county_csv)
    name_output_MPO_csv    = "".join(name_output_MPO_csv   )

if geography == 'MSA':
    name_output_MSA_csv = [indicator_name, '_MSA_', estimate, '.csv']
    name_output_MSA_csv = "".join(name_output_MSA_csv)

if geography == 'PUMA':
    name_output_PUMA_csv = [indicator_name, '_PUMA_', estimate, '.csv']
    name_output_PUMA_csv = "".join(name_output_PUMA_csv)


In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )

if geography == 'Tract':

    df_acs1_csv     .to_csv(os.path.join(path_out_csv, name_output_tract_csv ), index = False)
    df_counties1_csv.to_csv(os.path.join(path_out_csv, name_output_county_csv), index = False)
    df_mpo1_csv     .to_csv(os.path.join(path_out_csv, name_output_MPO_csv   ), index = False)

    if percentages == 'Yes':
        # Export long
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
            df_acs1     .to_excel(writer, index = False, sheet_name = 'Tracts'  )
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
            df_mpo1     .to_excel(writer, index = False, sheet_name = 'MPO'     )
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), engine='xlsxwriter') as writer:
            df_acs2          .to_excel(writer, index = False, sheet_name = 'Tracts Total'   )
            df_counties2     .to_excel(writer, index = False, sheet_name = 'Counties Total' )
            df_mpo2          .to_excel(writer, index = False, sheet_name = 'MPO Total'      )
            df_acs2_perc     .to_excel(writer, index = False, sheet_name = 'Tracts Perc'  )
            df_counties2_perc.to_excel(writer, index = False, sheet_name = 'Counties Perc')
            df_mpo2_perc     .to_excel(writer, index = False, sheet_name = 'MPO Perc'     )   
    
    else:
        # Export long
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
            df_acs1     .to_excel(writer, index = False, sheet_name = 'Tracts'  )
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
            df_mpo1     .to_excel(writer, index = False, sheet_name = 'MPO'     )
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), engine='xlsxwriter') as writer:
            df_acs2          .to_excel(writer, index = False, sheet_name = 'Tracts'   )
            df_counties2     .to_excel(writer, index = False, sheet_name = 'Counties' )
            df_mpo2          .to_excel(writer, index = False, sheet_name = 'MPO'      )


if geography == 'MSA':

    df_msa1_csv.to_csv(os.path.join(path_out_csv, name_output_MSA_csv), index = False)
    
    if percentages == 'Yes':
    
        # Export long
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
            df_msa1.to_excel(writer, index = False, sheet_name = 'MSA'     )
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), engine='xlsxwriter') as writer:
            df_msa2     .to_excel(writer, index = False, sheet_name = 'MSA Total')
            df_msa2_perc.to_excel(writer, index = False, sheet_name = 'MSA Perc')
    else:
        # Export long
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
            df_msa1.to_excel(writer, index = False, sheet_name = 'MSA'     )
        # Export wide
        with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_wide_xlsx), engine='xlsxwriter') as writer:
            df_msa2.to_excel(writer, index = False, sheet_name = 'MSA')


if geography == 'PUMA':

    df_acs_csv.to_csv(os.path.join(path_out_csv, name_output_PUMA_csv), index = False)
    
    # if percentages == 'Yes':
    #     # Export long
    #     with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
    #         df_acs.to_excel(writer, index = False, sheet_name = 'PUMS')        
    # else:
    #     # Export long
    #     with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_long_xlsx), engine='xlsxwriter') as writer:
    #         df_acs.to_excel(writer, index = False, sheet_name = 'PUMS')

print('')
print("Successfully exported")